In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from quinex_application.visualisation.quinex_visualisation import QuinexVisualisation
from quinex_application.visualisation.metadata_visualisation import VisualisationMetadata

In [ ]:
visualisation = QuinexVisualisation()
metadata = VisualisationMetadata("../data/quinex/df_quinex_metadata_new.parquet")

# **Load Data**

In [ ]:
df_paper = visualisation.load_df("../data/paper/paper_dataset.parquet")
df_paper = df_paper[df_paper["Quinex"] == True]

In [ ]:
metadata_df = visualisation.load_df("../data/quinex/df_quinex_metadata.parquet")
quinex_df = visualisation.load_df("../data/visualisation_files/df_technologies_nan.parquet")

In [ ]:
metadata_df = metadata_df.drop_duplicates(subset="DOI", keep="first")
len(metadata_df)

# **Analysis Metadata**

In [ ]:
fig, piv_entity, piv_property = metadata.mention_trend_heatmap(
    quinex_df[quinex_df["publication_year"] < 2026], 
    "entity_base_mapping", 
    "property_base_mapping", 
    figsize=(24,6), 
    font_size=30, 
    top_n=6, 
    cmap="Blues"
)
fig.savefig("../data/figures/metadata/ent_prop_distribution.pdf", bbox_inches='tight')

In [ ]:
eu_countries = {
    "EU": ["DE", "FR", "IT", "ES", "NL", "BE", "AT", "SE", "DK", "PL", "GR", "PT", "CZ", "FI", "RO", "HU", "BG", "HR", "SI", "SK", "IE"]
}

fig, pivot_countries, avg_sjr = metadata.countries_share_and_sjr(
    metadata_df,
    top_n_line = 5, 
    top_n_scatter = 50, 
    figsize = (24, 6), 
    font_size=30, 
    legend_bbox=(0.885,0.55),
    max_year=2024,
    normalize_by_population=True,
    population_excel_path="../data/visualisation_files/country_add_info/world_population.xlsx",
    # normalize_by_gdp=True,
    # gdp_excel_path="../data/visualisation_files/world_gdp.xls",
    country_groups=eu_countries
)
#fig.savefig("../data/figures/metadata/share_top_countries.pdf", bbox_inches='tight')

### **Country Comparison**

In [ ]:
# Create DataFrames for additional top research nations
def get_df_by_country(metadata_df, quinex_df, country_code):
    """Filter quinex_df for publications from a specific country."""
    pubs = metadata_df[metadata_df['countries'].apply(lambda x: country_code in x)]
    dois = pubs['DOI'].tolist()
    return quinex_df[quinex_df['DOI'].isin(dois)]

# Create EU DataFrame by aggregating EU countries
def get_df_by_countries(metadata_df, quinex_df, country_codes):
    """Filter quinex_df for publications from a list of countries."""
    pubs = metadata_df[metadata_df['countries'].apply(
        lambda x: any(code in x for code in country_codes)
    )]
    dois = pubs['DOI'].tolist()
    return quinex_df[quinex_df['DOI'].isin(dois)]


df_ch = get_df_by_country(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, "CN")  # China
df_us = get_df_by_country(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, "US")  # United States
df_jp = get_df_by_country(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, "JP")  # Japan
df_in = get_df_by_country(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, "IN")  # India
df_gb = get_df_by_country(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, "GB")  # United Kingdom

eu_codes = ["DE", "FR", "IT", "ES", "NL", "BE", "AT", "SE", "DK", "PL", "GR", "PT", "CZ", "FI", "RO", "HU", "BG", "HR", "SI", "SK", "IE"]
df_eu = get_df_by_countries(metadata_df[metadata_df["publication_year"] < 2025], quinex_df, eu_codes)

print(f"EU: {len(df_eu['DOI'].value_counts())} entries")
print(f"China: {len(df_ch['DOI'].value_counts())} entries")
print(f"USA: {len(df_us['DOI'].value_counts())} entries")
print(f"UK: {len(df_gb['DOI'].value_counts())} entries")
print(f"Japan: {len(df_jp['DOI'].value_counts())} entries")
print(f"India: {len(df_in['DOI'].value_counts())} entries")

In [ ]:
# 3 countries/regions: 3 columns, 1 row
fig, piv_entity = metadata.mention_trend_heatmap_multi_entities(
    dfs = [df_eu, df_ch, df_us, df_gb, df_jp, df_in],
    titles = [
        "EU: 9,653 Publications",
        "China: 5,591 Publications", 
        "USA: 2,895 Publications", 
        "UK: 2,596 Publications", 
        "Japan: 625 Publications", 
        "India: 1,372 Publications"
    ],
    entity_col = "entity_base_mapping",
    figsize=(24, 9),
    font_size=30, 
    top_n=8, 
    cmap="Blues",
    n_cols=3
)
fig.savefig("../data/figures/metadata/ent_6countries.pdf", bbox_inches='tight')